In [ ]:
# Module 5: Context Management & Multi-Agent Orchestration# Lab: Research & Synthesis Pipeline# Setup -- install dependencies (run once per session)# !pip install -q claude-agent-sdk python-dotenv

In [ ]:
# Import librariesimport osimport jsonimport asynciofrom pathlib import Pathfrom dotenv import load_dotenvfrom claude_agent_sdk import query, ClaudeAgentOptions

In [ ]:
# Load API keys from .env fileload_dotenv()ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")print(f"Anthropic key (SDK): {'Yes' if ANTHROPIC_API_KEY else 'No'}")

In [ ]:
# Step 1 -- Define the Researcher sub-agent# The Researcher only has WebSearch and WebFetch. No write access.async def run_researcher(topic: str) -> str:    """Gather research on a topic using web tools only."""    options = ClaudeAgentOptions(        allowed_tools=["WebSearch", "WebFetch"],    )    prompt = f"""Research the topic '{topic}' and return concise findings.Use WebSearch to find relevant information, then WebFetch to read details.Return a bullet-point summary of the most important facts only."""    result = ""    async for message in query(prompt=prompt, options=options):        if hasattr(message, 'content') and message.content:            result = message.content    return result

In [ ]:
# Step 2 -- Define the Writer sub-agent# The Writer only has the Edit tool. No web access.async def run_writer(findings: str, template_path: str, output_path: str) -> str:    """Write findings into the report template using Edit tool only."""    options = ClaudeAgentOptions(        allowed_tools=["Edit"],    )    prompt = f"""Read the template at {template_path}, then write a completedreport to {output_path} using the Edit tool.Findings to incorporate:{findings}Replace every placeholder in the template with real content.Do NOT modify any other files."""    result = ""    async for message in query(prompt=prompt, options=options):        if hasattr(message, 'content') and message.content:            result = message.content    return result

In [ ]:
# Step 3 -- Define the Coordinator# The Coordinator orchestrates the full pipeline: research -> writeasync def run_coordinator(task: str, template_path: str, output_path: str) -> str:    """Orchestrate research and writing phases."""    print("[Coordinator] Starting research phase...")    findings = await run_researcher(task)    print(f"[Coordinator] Research complete. {len(findings)} chars gathered.")    print("[Coordinator] Starting writing phase...")    report = await run_writer(findings, template_path, output_path)    print("[Coordinator] Report written.")    return report

In [ ]:
# Step 4 -- Execute the pipeline# Set target paths and run the full orchestration# Each sub-agent gets its own fresh context windowTEMPLATE_PATH = "data/report_template.md"OUTPUT_PATH = "data/completed_report.md"TASK = "Quantum Computing"result = await run_coordinator(TASK, TEMPLATE_PATH, OUTPUT_PATH)print("\n--- Final Report ---\n")print(result)

In [ ]:
# Step 5 -- Verify the output# Read the completed report to verify the Writer filled in the templatereport_file = Path(OUTPUT_PATH)if report_file.exists():    print("--- Completed Report ---")    print(report_file.read_text())else:    print("Report not found.")